# LeetCode 2399 — Check Distances Between Same Letters bằng Python

Link đề: https://leetcode.com/problems/check-distances-between-same-letters/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, String

Skeleton LeetCode:

```python
class Solution:
    def checkDistances(self, s: str, distance: List[int]) -> bool:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mỗi chữ cái trong `s` xuất hiện **đúng hai lần**, và khoảng cách "giữa hai lần" phải khớp với mảng `distance`.
2. Phân biệt **hiệu chỉ số** `j - i` và **số ký tự nằm giữa** `j - i - 1`.
3. Ánh xạ chữ cái sang chỉ số `0..25` bằng `ord(c) - ord('a')`, và dùng `dict` / mảng 26 phần tử để nhớ vị trí lần xuất hiện đầu.
4. Viết 4 cách: `index`/`rindex`, `dict` một lượt, mảng 26 vị trí, gom danh sách vị trí; đối chiếu bằng test ngẫu nhiên.
5. Tránh các lỗi: quên `- 1`, kiểm tra cả những chữ **không** có trong `s`, tính sai chỉ số chữ cái (`ord(c) - 96`).

## 2. Đề bài & ví dụ

Cho chuỗi `s` chỉ gồm chữ thường, trong đó **mỗi chữ cái xuất hiện đúng hai lần** (nếu nó xuất hiện), và một mảng `distance` độ dài 26.

Các chữ cái được đánh số từ `0` (`'a'`) đến `25` (`'z'`). Với mỗi chữ cái có mặt trong `s`, số ký tự nằm **giữa** hai lần xuất hiện của nó phải **bằng** `distance[chỉ số chữ cái]`. Với chữ cái **không** có trong `s`, bỏ qua `distance` tương ứng.

Trả về `True` nếu `s` là một chuỗi *well-spaced* (cách đều đúng yêu cầu), ngược lại `False`.

| Ví dụ | s | distance (các giá trị đầu) | Output |
|---|---|---|---|
| 1 | `"abaccb"` | `[1,3,0,5,0,0,...,0]` | `True` |
| 2 | `"aa"` | `[2,0,0,...,0]` | `False` |

Giải thích ví dụ 1: `'a'` ở chỉ số 0 và 2 → giữa có 1 ký tự = `distance[0] = 1`. `'b'` ở chỉ số 1 và 5 → giữa có 3 ký tự = `distance[1] = 3`. `'c'` ở chỉ số 3 và 4 → giữa có 0 ký tự = `distance[2] = 0`. Ví dụ 2: `'a'` ở 0 và 1 → giữa 0 ký tự nhưng `distance[0] = 2`.

Constraints: `2 <= s.length <= 52`; `s` chỉ gồm chữ thường; mỗi chữ cái trong `s` xuất hiện đúng hai lần; `distance.length == 26`; `0 <= distance[i] <= 50`.

In [1]:
s = "abaccb"
distance = [1, 3, 0, 5] + [0] * 22
print("len(distance) =", len(distance))
for c in sorted(set(s)):
    i, j = s.index(c), s.rindex(c)
    idx = ord(c) - ord('a')
    print(f"'{c}': vị trí {i} và {j}, giữa có {j - i - 1}, distance[{idx}] = {distance[idx]}")

len(distance) = 26
'a': vị trí 0 và 2, giữa có 1, distance[0] = 1
'b': vị trí 1 và 5, giữa có 3, distance[1] = 3
'c': vị trí 3 và 4, giữa có 0, distance[2] = 0


## 3. Kiến thức nền

### 3.1. `ord`, `chr` và chỉ số chữ cái

`ord('a') == 97`, `ord('b') == 98`... Hiệu `ord(c) - ord('a')` cho chỉ số từ `0` (`'a'`) đến `25` (`'z'`). `chr(97 + k)` làm ngược lại.

In [2]:
print(ord('a'), ord('b'), ord('z'))
print([ord(c) - ord('a') for c in "abz"])
print(chr(97), chr(97 + 25))

97 98 122
[0, 1, 25]
a z


### 3.2. `str.index`, `str.rindex`, `str.find`

- `s.index(c)`: vị trí **đầu tiên** của `c` (lỗi `ValueError` nếu không có).
- `s.rindex(c)`: vị trí **cuối cùng**.
- `s.find(c)` / `s.rfind(c)`: như trên nhưng trả `-1` nếu không có (không lỗi). Đây là chỗ dễ sinh bug ở mục 13.

In [3]:
s = "abaccb"
print(s.index('b'), s.rindex('b'), s.find('z'), s.rfind('z'))
try:
    s.index('z')
except ValueError as e:
    print("ValueError:", e)

1 5 -1 -1
ValueError: substring not found


### 3.3. `set` và vòng lặp qua các chữ **có mặt**

`set(s)` cho các ký tự khác nhau của `s`. Duyệt `set(s)` đảm bảo ta chỉ kiểm tra chữ cái có mặt, bỏ qua phần còn lại của `distance` (đúng yêu cầu đề).

In [4]:
s = "abaccb"
print(set(s), sorted(set(s)), len(set(s)))

{'b', 'c', 'a'} ['a', 'b', 'c'] 3


### 3.4. `dict` để nhớ vị trí đầu tiên

`first = {}` rồi `first[c] = i` khi gặp lần đầu. Khi gặp lần thứ hai, ta tra `first[c]` để tính khoảng cách. Kiểm tra tồn tại bằng `c in first`.

In [5]:
first = {}
for i, c in enumerate("abaccb"):
    if c not in first:
        first[c] = i
        print(f"i={i}: gặp '{c}' lần đầu, lưu {first}")
    else:
        print(f"i={i}: gặp '{c}' lần hai, giữa có {i - first[c] - 1} ký tự")

i=0: gặp 'a' lần đầu, lưu {'a': 0}
i=1: gặp 'b' lần đầu, lưu {'a': 0, 'b': 1}
i=2: gặp 'a' lần hai, giữa có 1 ký tự
i=3: gặp 'c' lần đầu, lưu {'a': 0, 'b': 1, 'c': 3}
i=4: gặp 'c' lần hai, giữa có 0 ký tự
i=5: gặp 'b' lần hai, giữa có 3 ký tự


### 3.5. Mảng 26 phần tử thay cho `dict`

Với bảng chữ cái cố định, `pos = [-1] * 26` rồi `pos[ord(c) - ord('a')]`. `-1` nghĩa là "chưa thấy". Nhanh và gọn; nhưng nhớ rằng `-1` là chỉ số hợp lệ của list Python (phần tử cuối) — luôn kiểm tra `== -1`, đừng dùng làm chỉ số.

In [6]:
pos = [-1] * 26
pos[ord('c') - ord('a')] = 3
print(pos[:5], pos[2])

[-1, -1, 3, -1, -1] 3


### 3.6. `all(...)` và thoát sớm

`all(biểu thức for ...)` trả `True` nếu mọi phần tử đúng và dừng ngay khi gặp `False`.

In [7]:
print(all(x > 0 for x in [1, 2, 3]), all(x > 0 for x in [1, -2, 3]))

True False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Chọn một chữ cái, đứng ở lần xuất hiện thứ nhất, đi tới lần xuất hiện thứ hai và **đếm số ký tự nằm giữa**. So con số này với `distance[chỉ số chữ cái]`. Nếu có chữ nào lệch → `False`; nếu mọi chữ đều khớp → `True`.

Tính tay ví dụ 1, `s = "abaccb"`:

```
chỉ số:  0 1 2 3 4 5
ký tự :  a b a c c b
a: 0 và 2 -> giữa = 2-0-1 = 1 ; distance[0]=1  ✓
b: 1 và 5 -> giữa = 5-1-1 = 3 ; distance[1]=3  ✓
c: 3 và 4 -> giữa = 4-3-1 = 0 ; distance[2]=0  ✓
```

Ba chữ cái đều khớp → `True`. Ví dụ 2 `"aa"`: giữa `= 1-0-1 = 0` mà `distance[0] = 2` → `False`.

Hai điểm tinh tế cần ghi nhớ:

- **Số ký tự giữa = `j - i - 1`**, không phải `j - i`.
- Các chữ cái không xuất hiện trong `s` thì `distance` của chúng **không cần** thỏa mãn gì cả.

## 5. Cách 1 — Brute Force / baseline

Với mỗi chữ cái có trong `s`, lấy lần xuất hiện đầu bằng `s.index(c)` và lần cuối bằng `s.rindex(c)`. Vì mỗi chữ cái xuất hiện đúng 2 lần nên "đầu" và "cuối" chính là hai lần đó.

Nút thắt: `index`/`rindex` quét lại chuỗi cho từng chữ cái (tối đa 26 lần quét) — vẫn nhanh với `n <= 52`, nhưng không tận dụng việc chỉ cần **một lượt duyệt**.

In [8]:
class SolutionBrute:
    def checkDistances(self, s, distance):
        for c in set(s):
            i = s.index(c)
            j = s.rindex(c)
            if j - i - 1 != distance[ord(c) - ord('a')]:
                return False
        return True

d1 = [1, 3, 0, 5] + [0] * 22
d2 = [2] + [0] * 25
print(SolutionBrute().checkDistances("abaccb", d1))
print(SolutionBrute().checkDistances("aa", d2))

True
False


**Độ phức tạp:** gọi `n = len(s)`; có tối đa 26 chữ cái khác nhau, mỗi chữ cái quét `O(n)` → `O(26·n) = O(n)` với hằng số 26. Space `O(26)` cho `set(s)`. Nút thắt: quét nhiều lần thay vì một lượt.

## 6. Ý tưởng thuật toán / bước đột phá

Chỉ cần **một lượt duyệt** từ trái sang phải với bảng "vị trí lần đầu":

- Nếu gặp chữ `c` lần đầu → ghi `first[c] = i`.
- Nếu gặp chữ `c` lần hai (đã có trong bảng) → tính `i - first[c] - 1` và so với `distance[...]`; lệch thì trả `False` ngay.

Vì mỗi chữ cái xuất hiện **đúng 2 lần**, mỗi chữ chỉ được kiểm tra một lần ở lần gặp thứ hai. Invariant: tại mọi thời điểm, `first` chứa vị trí lần đầu của **những chữ đã gặp đúng một lần**; khi một chữ được kiểm tra xong, bảng vẫn cho phép ta nhận biết "đã xử lý".

Quy tắc "nếu… thì…":

- Nếu `c` chưa có trong bảng → đây là lần đầu, ghi nhớ.
- Nếu `c` đã có → đây là lần hai: kiểm tra `i - first[c] - 1 == distance[idx]`.
- Duyệt hết mà không lệch → `True`.

In [9]:
s = "abaccb"
d = [1, 3, 0, 5] + [0] * 22
first = {}
ok = True
for i, c in enumerate(s):
    if c in first:
        giua = i - first[c] - 1
        want = d[ord(c) - ord('a')]
        print(f"i={i} '{c}': giữa={giua}, cần={want} -> {'khớp' if giua == want else 'LỆCH'}")
        ok = ok and giua == want
    else:
        first[c] = i
print("kết quả:", ok)

i=2 'a': giữa=1, cần=1 -> khớp
i=4 'c': giữa=0, cần=0 -> khớp
i=5 'b': giữa=3, cần=3 -> khớp
kết quả: True


## 7. Các cách tiếp theo

Ta có 4 cách: (1) `index`/`rindex`, (2) `dict` một lượt, (3) mảng 26 vị trí, (4) gom danh sách vị trí rồi kiểm tra.

### Cách 2 — `dict` một lượt

Đúng như mục 6: duyệt `enumerate(s)`, dùng `dict` lưu vị trí lần đầu.

In [10]:
class SolutionDict:
    def checkDistances(self, s, distance):
        first = {}
        for i, c in enumerate(s):
            if c in first:
                if i - first[c] - 1 != distance[ord(c) - ord('a')]:
                    return False
            else:
                first[c] = i
        return True

print(SolutionDict().checkDistances("abaccb", d1))
print(SolutionDict().checkDistances("aa", d2))

True
False


Độ phức tạp Cách 2: Time `O(n)`, Space `O(26) = O(1)` (tối đa 26 khoá).

### Cách 3 — Mảng 26 phần tử `pos`

Thay `dict` bằng list `pos = [-1] * 26`. Dùng `-1` làm "chưa thấy". Phù hợp khi bảng ký tự cố định và nhỏ, tránh băm.

In [11]:
class SolutionArray:
    def checkDistances(self, s, distance):
        pos = [-1] * 26
        for i, c in enumerate(s):
            k = ord(c) - ord('a')
            if pos[k] == -1:
                pos[k] = i
            elif i - pos[k] - 1 != distance[k]:
                return False
        return True

print(SolutionArray().checkDistances("abaccb", d1))
print(SolutionArray().checkDistances("aa", d2))

True
False


Độ phức tạp Cách 3: Time `O(n)`, Space `O(26) = O(1)`.

### Cách 4 — Gom danh sách vị trí rồi kiểm tra

Đầu tiên gom `where[c] = [các vị trí]` cho mỗi chữ, sau đó kiểm tra `where[c][1] - where[c][0] - 1`. Cách này **tách hai pha** (gom rồi kiểm tra), dễ debug và dễ mở rộng nếu đề đổi thành "xuất hiện nhiều lần".

In [12]:
from collections import defaultdict

class SolutionGroup:
    def checkDistances(self, s, distance):
        where = defaultdict(list)
        for i, c in enumerate(s):
            where[c].append(i)
        return all(p[-1] - p[0] - 1 == distance[ord(c) - ord('a')] for c, p in where.items())

print(SolutionGroup().checkDistances("abaccb", d1))
print(SolutionGroup().checkDistances("aa", d2))

True
False


Độ phức tạp Cách 4: Time `O(n)`, Space `O(n)` (lưu mọi vị trí).

## 8. TRACE từng bước

Trace Cách 3 với `s = "abaccb"` và `distance = [1,3,0,...]`.

| i | c | k | `pos[k]` trước | hành động | giữa | cần |
|---|---|---|---|---|---|---|
| 0 | a | 0 | -1 | lưu `pos[0]=0` | – | – |
| 1 | b | 1 | -1 | lưu `pos[1]=1` | – | – |
| 2 | a | 0 | 0 | kiểm tra | 1 | 1 ✓ |
| 3 | c | 2 | -1 | lưu `pos[2]=3` | – | – |
| 4 | c | 2 | 3 | kiểm tra | 0 | 0 ✓ |
| 5 | b | 1 | 1 | kiểm tra | 3 | 3 ✓ |

In [13]:
def trace_array(s, distance):
    pos = [-1] * 26
    for i, c in enumerate(s):
        k = ord(c) - ord('a')
        if pos[k] == -1:
            pos[k] = i
            print(f"i={i} c={c} k={k}: lần đầu, pos[{k}] = {i}")
        else:
            giua, can = i - pos[k] - 1, distance[k]
            print(f"i={i} c={c} k={k}: lần hai, giữa={giua}, cần={can} -> {'OK' if giua == can else 'LỆCH, dừng'}")
            if giua != can:
                return False
    return True

print(trace_array("abaccb", d1))
print(trace_array("aa", d2))

i=0 c=a k=0: lần đầu, pos[0] = 0
i=1 c=b k=1: lần đầu, pos[1] = 1
i=2 c=a k=0: lần hai, giữa=1, cần=1 -> OK
i=3 c=c k=2: lần đầu, pos[2] = 3
i=4 c=c k=2: lần hai, giữa=0, cần=0 -> OK
i=5 c=b k=1: lần hai, giữa=3, cần=3 -> OK
True
i=0 c=a k=0: lần đầu, pos[0] = 0
i=1 c=a k=0: lần hai, giữa=0, cần=2 -> LỆCH, dừng
False


## 9. Code nộp LeetCode

In [14]:
from typing import List

class Solution:
    def checkDistances(self, s: str, distance: List[int]) -> bool:
        pos = [-1] * 26
        for i, c in enumerate(s):
            k = ord(c) - ord('a')
            if pos[k] == -1:
                pos[k] = i
            elif i - pos[k] - 1 != distance[k]:
                return False
        return True

print(Solution().checkDistances("abaccb", d1), Solution().checkDistances("aa", d2))

True False


## 10. Vì sao thuật toán đúng

- **Mỗi chữ được kiểm tra đúng một lần:** do mỗi chữ có đúng hai vị trí, lần gặp thứ hai là lúc ta biết cả hai vị trí và tính chính xác `j - i - 1`.
- **Không bỏ sót:** mọi chữ trong `s` đều có lần gặp thứ hai nên không chữ nào thoát khỏi kiểm tra.
- **Không kiểm tra thừa:** những chữ cái không có trong `s` không bao giờ được chạm tới, đúng như đề ("bỏ qua").
- **Khớp định nghĩa:** số ký tự **giữa** hai chỉ số `i < j` là `j - i - 1` (các chỉ số `i+1..j-1`). Trả `False` sớm là hợp lệ vì chỉ cần một chữ lệch là chuỗi không còn *well-spaced*.
- **Thoát sớm vẫn đúng:** `False` khi một điều kiện trong phép hội (`AND`) sai; ngược lại, chỉ khi duyệt xong mà không lệch mới trả `True`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)` (`<= 52`). Số chữ cái tối đa `26`.

| Cách | Time | Space |
|---|---|---|
| 1. `index`/`rindex` | `O(26·n)` | `O(26)` |
| 2. `dict` một lượt | `O(n)` | `O(26)` |
| 3. mảng 26 | `O(n)` | `O(26)` |
| 4. gom danh sách | `O(n)` | `O(n)` |

Vì alphabet cố định, mọi cách đều coi là `O(n)` thời gian; Space `O(1)` với cách 2, 3. Không có khác biệt average/worst (luôn duyệt tối đa hết chuỗi; thoát sớm chỉ tốt hơn).

## 12. Bộ test

Hàm tiện ích `mk` tạo mảng `distance` 26 phần tử từ một dict `{chữ cái: khoảng cách}`, các phần tử còn lại cho giá trị bất kỳ (ở đây `0` hoặc một số "bẫy").

In [15]:
import random, string

def mk(d, fill=0):
    arr = [fill] * 26
    for c, v in d.items():
        arr[ord(c) - ord('a')] = v
    return arr

impls = [SolutionBrute(), SolutionDict(), SolutionArray(), SolutionGroup(), Solution()]
tests = [
    ("abaccb", mk({'a': 1, 'b': 3, 'c': 0}), True),
    ("aa", mk({'a': 2}), False),
    ("aa", mk({'a': 0}), True),
    ("aa", mk({'a': 0}, fill=50), True),                   # chữ vắng mặt có distance lớn vẫn bị bỏ qua
    ("abab", mk({'a': 1, 'b': 1}), True),
    ("abba", mk({'a': 2, 'b': 0}), True),
    ("abba", mk({'a': 0, 'b': 0}), False),
    ("zz", mk({'z': 0}), True),
    ("azaz", mk({'a': 1, 'z': 1}), True),
    ("abcabc", mk({'a': 2, 'b': 2, 'c': 2}), True),
    ("abcabc", mk({'a': 2, 'b': 2, 'c': 1}), False),
]
for s, d, exp in tests:
    for im in impls:
        assert im.checkDistances(s, d) is exp, (type(im).__name__, s, exp)
print("tất cả test cố định OK:", len(tests))

tất cả test cố định OK: 11


Test 2: ngẫu nhiên có seed, đúng constraints (mỗi chữ trong `s` xuất hiện **đúng 2 lần**, `2 <= len(s) <= 52`, `0 <= distance[i] <= 50`). Một nửa số test được xây từ khoảng cách thật nên đáp án `True` xuất hiện thường xuyên; nửa còn lại ngẫu nhiên hoàn toàn. Đối chiếu với một oracle độc lập dùng `find`.

In [16]:
random.seed(2399)

def oracle(s, d):
    for c in string.ascii_lowercase:
        if c in s:
            i = s.find(c)
            j = s.find(c, i + 1)
            if j - i - 1 != d[ord(c) - ord('a')]:
                return False
    return True

cnt_true = 0
for t in range(4000):
    m = random.randint(1, 26)
    letters = random.sample(string.ascii_lowercase, m)
    chars = [c for c in letters for _ in range(2)]
    random.shuffle(chars)
    s = ''.join(chars)
    assert 2 <= len(s) <= 52 and all(s.count(c) == 2 for c in set(s))
    d = [random.randint(0, 50) for _ in range(26)]
    if t % 2 == 0:                                   # làm đúng khoảng cách thật cho các chữ có mặt
        for c in set(s):
            d[ord(c) - ord('a')] = s.rindex(c) - s.index(c) - 1
        if random.random() < 0.3:                    # rồi phá hỏng ngẫu nhiên một chữ
            c = random.choice(list(set(s)))
            d[ord(c) - ord('a')] = random.randint(0, 50)
    exp = oracle(s, d)
    cnt_true += exp
    for im in impls:
        assert im.checkDistances(s, d) is exp, (type(im).__name__, s, d)
print("4000 test ngẫu nhiên OK, số test đáp án True:", cnt_true)
assert 0 < cnt_true < 4000

4000 test ngẫu nhiên OK, số test đáp án True: 1398


Test 3: test lớn nhất theo đề (`s` có đủ 26 chữ cái, 52 ký tự) và hai chuỗi đặc biệt.

In [17]:
s_big = string.ascii_lowercase * 2                       # mỗi chữ cách nhau 25 ký tự giữa
d_big = [25] * 26
assert all(im.checkDistances(s_big, d_big) is True for im in impls)
s_nest = string.ascii_lowercase + string.ascii_lowercase[::-1]   # a...z z...a: lồng nhau
d_nest = [2 * (25 - k) for k in range(26)]
assert all(im.checkDistances(s_nest, d_nest) is True for im in impls)
d_nest[3] += 1
assert all(im.checkDistances(s_nest, d_nest) is False for im in impls)
print("test biên 26 chữ OK")

test biên 26 chữ OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: quên `- 1` (nhầm hiệu chỉ số với số ký tự giữa)

In [18]:
def wrong_no_minus1(s, distance):
    for c in set(s):
        if s.rindex(c) - s.index(c) != distance[ord(c) - ord('a')]:   # SAI: thiếu - 1
            return False
    return True

print("sai :", wrong_no_minus1("abaccb", d1))
print("đúng:", Solution().checkDistances("abaccb", d1))
assert wrong_no_minus1("abaccb", d1) != Solution().checkDistances("abaccb", d1)

sai : False
đúng: True


Vì sao sai: `'a'` ở chỉ số 0 và 2 có hiệu `2` nhưng giữa chỉ có **1** ký tự. Đề dùng "số ký tự giữa", nên phải trừ 1.

### Lỗi 2: kiểm tra cả 26 chữ, kể cả chữ không có trong `s`

In [19]:
def wrong_all26(s, distance):
    for k in range(26):
        c = chr(97 + k)
        i, j = s.find(c), s.rfind(c)           # chữ vắng mặt -> i = j = -1
        if j - i - 1 != distance[k]:           # SAI: -1 - (-1) - 1 = -1 != 0
            return False
    return True

print("sai :", wrong_all26("aa", mk({'a': 0})))
print("đúng:", Solution().checkDistances("aa", mk({'a': 0})))
assert wrong_all26("aa", mk({'a': 0})) != Solution().checkDistances("aa", mk({'a': 0}))

sai : False
đúng: True


Vì sao sai: với chữ vắng mặt, `find`/`rfind` đều trả `-1`, công thức cho `-1`, luôn khác `distance` (không âm) → luôn trả `False`. **Sửa:** chỉ xét các chữ **có trong `s`** (`set(s)`) hoặc kiểm tra `c in s` trước.

### Lỗi 3: ánh xạ chữ cái bị lệch 1 (`ord(c) - 96`)

In [20]:
def wrong_offset(s, distance):
    first = {}
    for i, c in enumerate(s):
        k = ord(c) - 96                       # SAI: 'a' -> 1, không phải 0
        if c in first:
            if i - first[c] - 1 != distance[k]:
                return False
        else:
            first[c] = i
    return True

print("sai :", wrong_offset("abaccb", d1))
print("đúng:", Solution().checkDistances("abaccb", d1))
assert wrong_offset("abaccb", d1) != Solution().checkDistances("abaccb", d1)

sai : False
đúng: True


Vì sao sai: `'a'` phải ứng với `distance[0]`; `ord(c) - 96` cho `1` nên lấy nhầm `distance` của chữ `'b'`. Với `'z'` còn gây `IndexError` vì chỉ số `26` ngoài mảng 26 phần tử. **Sửa:** `ord(c) - ord('a')`.

### Lỗi 4 (bonus): dùng `-1` làm chỉ số

`pos[-1]` không báo lỗi mà trả phần tử **cuối** của list. Nếu bạn viết `if pos[k]:` để kiểm tra "đã thấy", thì vị trí `0` (falsy) bị coi như "chưa thấy". Hãy so sánh tường minh `pos[k] == -1`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Hai lần xuất hiện của cùng một ký tự" + "khoảng cách" → duyệt một lượt, lưu **vị trí lần đầu** (`dict` hoặc mảng 26).
- Bảng chữ cái cố định `a..z` → mảng 26 phần tử với `ord(c) - ord('a')`.
- Khoảng cách **giữa** hai chỉ số `i < j` là `j - i - 1`; khoảng cách **chỉ số** là `j - i`. Đọc kỹ đề dùng cái nào.
- Khi đề nói "bỏ qua chữ không có trong `s`", đừng lặp qua cả `distance` mà hãy lặp qua `s`.
- Hãy viết sẵn hàm tạo input `mk(...)` để test nhanh, tránh gõ tay mảng 26 phần tử.
- Mỗi chữ chỉ xuất hiện đúng 2 lần nên không cần `Counter`; nhưng nếu đề đổi thì cách 4 (gom vị trí) mở rộng dễ nhất.

## 15. Biến thể của bài

1. Mỗi chữ có thể xuất hiện **nhiều lần** (khoảng cách giữa các lần liên tiếp phải bằng nhau / bằng một giá trị).
2. Trả về **danh sách chữ cái vi phạm** thay vì `True/False`.
3. Tìm hai lần xuất hiện **gần nhất** của cùng một chữ (bài "nearest same character").
4. Với mảng số: `nums[i] == nums[j]` và `abs(i - j) <= k` (Contains Duplicate II).
5. Cho `s` và hỏi: có tồn tại `distance` làm `s` hợp lệ không? (luôn có, bằng cách đặt chính khoảng cách thực).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `index`/`rindex` | quét tìm đầu/cuối cho từng chữ | `O(26·n)` | `O(26)` | cực ngắn, dễ hiểu |
| 2. `dict` một lượt | nhớ vị trí đầu, kiểm tra lần hai | `O(n)` | `O(26)` | linh hoạt với mọi ký tự |
| 3. mảng 26 | như `dict` nhưng chỉ số bằng `ord` | `O(n)` | `O(26)` | **chuẩn** cho chữ thường |
| 4. gom danh sách | gom mọi vị trí rồi kiểm tra | `O(n)` | `O(n)` | dễ mở rộng nhiều lần xuất hiện |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `s = "abab"`, `distance` có `distance[0] = 1`, `distance[1] = 1` (phần còn lại là 0). Kết quả?

**Bài 2 (tính tay):** `s = "abcabc"` thì số ký tự giữa của `'a'`, `'b'`, `'c'` lần lượt là bao nhiêu?

**Bài 3 (điền TODO):** hoàn thành `between(s, c)` trả về số ký tự giữa hai lần xuất hiện của `c` trong `s`.

**Bài 4 (tự viết lại):** viết `wrong_letters(s, distance)` trả về **list các chữ cái** (theo thứ tự bảng chữ cái) mà khoảng cách không khớp.

**Bài 5 (mini challenge):** viết `ideal_distance(s)` trả về mảng 26 phần tử làm `s` hợp lệ (chữ vắng mặt đặt `0`), rồi dùng nó để kiểm tra `Solution().checkDistances(s, ideal_distance(s))` luôn `True`.

In [21]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().checkDistances("abab", mk({'a': 1, 'b': 1})))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def between(s, c):
    # TODO: trả về số ký tự nằm giữa hai lần xuất hiện của c
    pass

True


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** số ký tự giữa = `chỉ số lần hai - chỉ số lần đầu - 1`.

**Mức 2 (vừa):** Bài 3: `s.rindex(c) - s.index(c) - 1`. Bài 4: duyệt `sorted(set(s))`, so sánh với `distance[ord(c) - ord('a')]`, gom chữ lệch. Bài 5: đặt `d[ord(c)-97] = between(s, c)` cho mỗi chữ trong `s`.

**Mức 3 (gần đáp án):** Bài 4: `[c for c in sorted(set(s)) if between(s, c) != distance[ord(c) - ord('a')]]`. Bài 5: tạo `d = [0] * 26`, lặp `for c in set(s): d[ord(c) - ord('a')] = between(s, c)`; trả `d`.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1: 'a' ở 0 và 2 (giữa 1), 'b' ở 1 và 3 (giữa 1) -> True
assert Solution().checkDistances("abab", mk({'a': 1, 'b': 1})) is True

# Bài 2: "abcabc" -> mỗi chữ cách nhau 2 ký tự giữa
assert [("abcabc".rindex(c) - "abcabc".index(c) - 1) for c in "abc"] == [2, 2, 2]

# Bài 3
def between(s, c):
    return s.rindex(c) - s.index(c) - 1
assert between("abaccb", 'b') == 3
assert between("aa", 'a') == 0

# Bài 4
def wrong_letters(s, distance):
    return [c for c in sorted(set(s)) if between(s, c) != distance[ord(c) - ord('a')]]
assert wrong_letters("abaccb", mk({'a': 1, 'b': 3, 'c': 0})) == []
assert wrong_letters("abaccb", mk({'a': 0, 'b': 3, 'c': 5})) == ['a', 'c']

# Bài 5
def ideal_distance(s):
    d = [0] * 26
    for c in set(s):
        d[ord(c) - ord('a')] = between(s, c)
    return d
for s in ("abaccb", "aa", "abcabc", "zyxzxy"):
    assert Solution().checkDistances(s, ideal_distance(s)) is True
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "nhớ vị trí lần đầu, kiểm tra lần hai":**

```python
pos = [-1] * 26
for i, c in enumerate(s):
    k = ord(c) - ord('a')
    if pos[k] == -1:
        pos[k] = i                     # lần đầu: lưu
    elif i - pos[k] - 1 != distance[k]:
        return False                   # lần hai: kiểm tra
return True
```

**Checklist trước khi submit:**

- [ ] Đang tính **số ký tự giữa** `j - i - 1`, không phải `j - i`?
- [ ] Chỉ kiểm tra các chữ **có mặt** trong `s`?
- [ ] Chỉ số chữ cái là `ord(c) - ord('a')` (bắt đầu từ 0)?
- [ ] Kiểm tra `pos[k] == -1` tường minh, không dựa vào truthy/falsy?
- [ ] Đã thử `"aa"` với `distance[0] = 0` và `distance[0] > 0`?
- [ ] Đã đối chiếu các cách bằng test ngẫu nhiên có seed (đúng constraint: mỗi chữ đúng 2 lần)?
- [ ] Chữ ký đúng `checkDistances(self, s: str, distance: List[int]) -> bool`?